In [10]:
import cv2,os,numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split


In [16]:
x = []
y = []
for classe in ["Chats","Chiens"]:
    dossier = os.path.join("datasets/train",classe)
    for fichier in tqdm(os.listdir(dossier)):
        chemin = os.path.join(dossier,fichier)
        image = cv2.imread(chemin)
        image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        image = cv2.resize(image,(64,64))
        ## Normalisation
        image = image/255.0
        ## flatten
        ## mes images sont de tailles 64x64x3
        x.append(image)

        if classe == "Chats":
            y.append(0)
        else:
            y.append(1)

x = np.array(x)
y = np.array(y)


100%|██████████| 12500/12500 [00:47<00:00, 261.99it/s]


In [17]:
print(x.shape)
## Le shape de x correspond a 25000 images de chat et des chiens, de dimensions 64*64*3
print(y.shape)
## y represnte les classes, "0" pour chat et "1" pour chien

(25000, 64, 64, 3)
(25000,)


In [18]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (20000, 64, 64, 3)
X_test: (5000, 64, 64, 3)
y_train: (20000,)
y_test: (5000,)


In [19]:
## je les convertis en float 32 pour qu'elles prennent moins de mémoire lors de l'exécution

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

In [30]:
## Je n'ai pas obtenu de grands résultats avec le réseaux de neurones connectés, j'utilise désormais les CNN
from tensorflow.keras import layers,models
model = models.Sequential()
## je cree ma premiere couche de convolution
## Je cree une couche de convoltion de 32 filtres, d'un kernel de dimension (3,3) pour des images de tailles 64x64x3 
## *3 represnte les canaux(RGB) et ma fonction d'activation relu, pour que le modele apprenne des relations complexes, pas linéaires
## Le padding représente la dimension de ma matrice de sortie, juste apres convoltion. En mettant padding="same" on concerve le max d'informations
model.add(layers.Conv2D(32,(3,3),activation="relu",padding='same',input_shape = (64,64,3)))

## Le MaxPooling utilise une fenetre de 2x2 pour résuire la dimension du résultat de la convolution
## Après le MaxPooling2D on aura une matrice de taille/2 c'est a dire 64/2=32 et de profondeur 32 filtres
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(64,(3,3),activation='relu',padding='same'))
## La on applique 64 filtres et on force le resultat a etre de la même dimension que l'entrée
## On aura donc une matrice de taille 32 et de profondeur 64
## Chaque nouveau filtres utilise toutes les maps precedentes pour produires une seule map
model.add(layers.MaxPooling2D(2,2))
## La on a des matrices de tailles 32/2=16 et de profondeur 64

model.add(layers.Conv2D(128,(3,3),activation='relu',padding='same'))
model.add(layers.MaxPooling2D(2,2))
## ma taille est de 8x8 et de profondeur 128

model.add(layers.Conv2D(256,(3,3),activation='relu',padding='same'))
model.add(layers.MaxPooling2D(2,2))
## ma taille est de dimension 4x4 et de profondeur 256

## L'intéret de reduire la taille tout en augmentant les features est de diminuer au maximum les données d'entrées de mon futur réseau tout en conservant les caractéristiques(filtres) importants
## J'applaties maintenant mes données pour avoir un seul vecteur que je vais faire passer dans un réseau de neurones profonds
model.add(layers.Flatten())


## je cree mon reseau de neurones profonds
model.add(layers.Dense(128,activation='relu'))
model.add(layers.Dropout(0.4))
model.add(layers.Dense(128,activation='relu'))
model.add(layers.Dropout(0.4))
model.add(layers.Dense(1,"sigmoid"))


In [31]:
## L'entrainemet
model.compile(optimizer='adam',## optimizer represente l'algorithme utilisé pour mettre a jour les poids, par ex descente de gradient, ect
              loss = 'binary_crossentropy', ## la fonction qui calcule le score ou l'erreur
              metrics = ['accuracy'])## Permet a l'humain de voir l'exactitude de ces résultats

In [32]:
history = model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=32)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 64s 93ms/step - accuracy: 0.5461 - loss: 0.6816 - val_accuracy: 0.5892 - val_loss: 0.6655
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 53s 84ms/step - accuracy: 0.6673 - loss: 0.6129 - val_accuracy: 0.7092 - val_loss: 0.5595
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 53s 84ms/step - accuracy: 0.7469 - loss: 0.5166 - val_accuracy: 0.7478 - val_loss: 0.5088
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.7958 - loss: 0.4421 - val_accuracy: 0.8018 - val_loss: 0.4343
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy: 0.8322 - loss: 0.3814 - val_accuracy: 0.8342 - val_loss: 0.3793
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.8547 - loss: 0.3331 - val_accuracy: 0.8424 - val_loss: 0.3551
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.8725 - loss: 0.2995 - val_accuracy: 0.8350 - val_loss: 0.3646
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 49s 78ms/step - accuracy: 0.8920 - loss: 0.2584 - 

In [33]:
## Je sauvegarde mon modele
model.save("cats_dogs.keras")